In [0]:
fk_zoo_id_sub_df = spark.sql(f"""
    SELECT fk_tvid
    , ts_start
    , ts_end
    , fk_zoo_id
    FROM (
        SELECT z.fk_tvid
        , v.ts_start
        , v.ts_end
        , z.fk_zoo_id
        , ROW_NUMBER() OVER (PARTITION BY z.fk_tvid, v.ts_start, v.ts_end ORDER BY z.create_timestamp DESC) AS fk_tvid_rank
        FROM {catalog_temp}.{db_name_temp}.staging_df v
        INNER JOIN detection.tv t 
          ON t.vizio_tvid = TRIM(v.tvid)
        INNER JOIN detection.tv_zoo z
          ON z.fk_tvid = t.tvid 
        AND v.ts_end > z.create_timestamp
        AND z.next_create_timestamp > DATE_SUB(current_date(), 5) -- 5 day interval in Photon
    ) a
    WHERE fk_tvid_rank = 1
""")

In [0]:
fk_input_source_id_sub_df = spark.sql(f"""
    SELECT fk_tvid, ts_start, ts_end, fk_input_source_id
    FROM (
        SELECT z.fk_tvid
        , z.fk_input_source_id
        , v.ts_start
        , v.ts_end
        , ROW_NUMBER() OVER (PARTITION BY z.fk_tvid, v.ts_start, v.ts_end ORDER BY z.create_timestamp DESC) AS fk_input_source_id_rank
        FROM {catalog_temp}.{db_name_temp}.staging_df v
        INNER JOIN detection.tv t 
          ON t.vizio_tvid = TRIM(v.tvid)
        INNER JOIN detection.tv_inputsource AS z 
          ON z.fk_tvid = t.tvid 
         AND v.ts_start >= z.create_timestamp -- 110 seconds INTERVAL removed to use photon
         AND v.ts_start< z.next_create_timestamp
         AND z.next_create_date > DATE_SUB(current_date(), 5)
    ) a
    WHERE a.fk_input_source_id_rank = 1;
""")

In [0]:
fk_location_id_sub_df = spark.sql(f"""
    SELECT fk_tvid, ts_start, ts_end,
    fk_location_id
    FROM (
        SELECT z.fk_tvid
        , v.ts_start
        , v.ts_end
        , z.fk_location_id
        , ROW_NUMBER() OVER (PARTITION BY  z.fk_tvid, v.ts_start, v.ts_end ORDER BY z.create_timestamp DESC) AS fk_location_id_rank
        FROM {catalog_temp}.{db_name_temp}.staging_df v
        INNER JOIN detection.tv t 
        ON t.vizio_tvid = TRIM(v.tvid)
        INNER JOIN detection.tv_geolocation z 
        ON z.fk_tvid = t.tvid 
        AND v.ts_end > z.create_timestamp
        AND z.next_create_date > DATE_SUB(current_date(), 5) -- replace - 5 day INTERVAL for photon
    ) a
    WHERE a.fk_location_id_rank = 1;
""")

In [0]:
def controllerFunction(staging_df, microBatchId):
    
    deltaHelpers.removeAllTempTablesForSession()
    deltaHelpers.saveToDeltaTempTable(staging_df, "staging_df")

    ########### Get Subquery data Frames

    minBatchDate = spark.sql(f"""SELECT MIN(ts_start) - interval 1 day FROM {catalog_temp}.{db_name_temp}.staging_df""").collect()[0][0]
    maxBatchDate = spark.sql(f"""SELECT MAX(ts_start) FROM {catalog_temp}.{db_name_temp}.staging_df""").collect()[0][0]


    


    ##### fk_input_source_id_sub_df
    

    ##### fk_location_id_sub_df
    

    ##### Persist to Delta Temp Tables to simplify execution plan

    fk_zoo_id_sub_df = deltaHelpers.saveToDeltaTempTable(fk_zoo_id_sub_df, "fk_zoo_id_sub_df")
    fk_input_source_id_sub_df = deltaHelpers.saveToDeltaTempTable(fk_input_source_id_sub_df, "fk_input_source_id_sub_df")
    fk_location_id_sub_df = deltaHelpers.saveToDeltaTempTable(fk_location_id_sub_df, "fk_location_id_sub_df")

    inscape_station_map_df = spark.sql(f"""
      SELECT 
        inscape_station_id, 
        inscape_call_sign, 
        mapped_vendor,
        mapped_vendor_station_id
    FROM (
        SELECT 
          inscape_station_id, 
          inscape_call_sign, 
          mapped_vendor, 
          mapped_vendor_station_id,
          ROW_NUMBER() OVER (PARTITION BY mapped_vendor, mapped_vendor_station_id ORDER BY created_at DESC) AS rn
        FROM detection.inscape_station_map
      ) ism
    WHERE ism.rn = 1
  """)

    deltaHelpers.saveToDeltaTempTable(inscape_station_map_df, "inscape_station_map_df")

    epg_schedule_latest_df = spark.sql(f"""
      SELECT 
        sch.fk_show_id, 
        sch.fk_station_id, 
        sch.schedule_id,
        sch.airdate, 
        sch.airdate_end, 
        sch.duration, 
        sch.vendor_name, 
        ism.inscape_station_id
      FROM detection.epg_schedule_latest sch
      JOIN {catalog_temp}.{db_name_temp}.inscape_station_map_df ism
        ON ism.mapped_vendor_station_id = sch.fk_station_id
       AND ism.mapped_vendor = sch.vendor_name
      WHERE DATE(sch.airdate) >= CURRENT_DATE - INTERVAL 120 DAY
        AND sch.airdate <= CURRENT_DATE + INTERVAL 2 DAY
      GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
      """)

    deltaHelpers.saveToDeltaTempTable(epg_schedule_latest_df, "epg_schedule_latest_df")


    ########### Parent Query Stage 1 temp table

    df_step_1 = spark.sql(f"""
        SELECT /*+ SKEW('c', 'content_cid') */
           t.tvid AS fk_tvid,
           e.show_id AS fk_show_id,
           map.inscape_station_id AS fk_station_id,
           CASE WHEN v.air_date LIKE '20%' THEN v.air_date::timestamp ELSE NULL END AS airdate,
           v.ts_start as session_start,
           v.ts_end as session_end,
           0 AS session_duration,
           v.mt_start AS media_time_start,
           v.mt_end AS media_time_end,
           sh.duration AS runtime,
           f.frame_id AS fk_frame_id,
           fk_sub1.fk_zoo_id AS fk_zoo_id,
           CASE WHEN v.cid = 'unknown' THEN {unknown_content_id}::integer ELSE c.content_id END AS fk_content_id,
           fk_sub2.fk_input_source_id AS fk_input_source_id,
           fk_sub3.fk_location_id AS fk_location_id,
           sh.schedule_id AS fk_schedule_id,
           v.is_live AS is_live,
           v.file_ingested AS file_ingested,
           v.confidence AS confidence,
           v.ump_id AS ump_id,
           v.batch_size AS batch_size,
           v.audio_contri AS audio_contri,
           v.video_contri AS video_contri,
           v.created_at AS created_at,
           DATE_TRUNC('HOUR', v.ts_start) AS session_hour, 
           DATE(v.ts_start) AS partition_key,
           v.input_file_name AS input_file_name
        FROM {catalog_temp}.{db_name_temp}.staging_df v
        INNER JOIN detection.tv t ON t.vizio_tvid = v.tvid --remove trim cause its an integer now
        INNER JOIN detection.frame_sizes_firehose f ON f.fr_w=v.fr_w 
            AND f.fr_h=v.fr_h 
            AND f.fr_rate=v.fr_rate
        LEFT JOIN detection.epg_show e 
          ON e.database_key = v.epid 
          AND e.vendor_name = 'TIVO'
        LEFT JOIN {catalog_temp}.{db_name_temp}.inscape_station_map_df map 
          ON map.inscape_call_sign = v.chan_callsign  -- Changed to chan_callsign from station_call_sign
          AND e.vendor_name = map.mapped_vendor
        LEFT JOIN detection.epg_schedule sh 
          ON map.mapped_vendor_station_id=sh.fk_station_id
          AND e.show_id=sh.fk_show_id
          AND e.vendor_name = sh.vendor_name
          AND sh.airdate=CASE WHEN v.air_date LIKE '20%' THEN v.air_date::timestamp ELSE NULL END
        LEFT JOIN detection.content_ids_firehose c ON c.content_cid=v.cid 
                                                    AND c.content_cid !='unknown' --Cody Added this cause it minimizes crazy skew 
                                                    AND COALESCE(c.fk_schedule_id,  -11111)= COALESCE(sh.schedule_id, -11111)
        LEFT JOIN {catalog_temp}.{db_name_temp}.fk_zoo_id_sub_df fk_sub1 ON fk_sub1.fk_tvid = t.tvid AND fk_sub1.ts_start = v.ts_start AND fk_sub1.ts_end = v.ts_end
        LEFT JOIN {catalog_temp}.{db_name_temp}.fk_input_source_id_sub_df fk_sub2 ON fk_sub2.fk_tvid = t.tvid AND fk_sub2.ts_start = v.ts_start AND fk_sub2.ts_end = v.ts_end
        LEFT JOIN {catalog_temp}.{db_name_temp}.fk_location_id_sub_df fk_sub3 ON fk_sub3.fk_tvid = t.tvid AND fk_sub3.ts_start = v.ts_start AND fk_sub3.ts_end = v.ts_end
    """)

  
    df_step_1 = deltaHelpers.saveToDeltaTempTable(df_step_1, "step_1_df")

    ##### NEW BASE TABLE ADDITIONS 2/16/2022
    ### New modification to remove simulcast shows to reduce duplicates

    # simulcast_shows = spark.sql(f"""WITH main_feed AS (
    #     SELECT DISTINCT
    #     sch.fk_show_id
    #     , sch.airdate
    #     , st.isn_affil AS station_name
    #     , LOWER(st.nat_or_loc) AS natloc
    #     , NVL(st.st_dma_id, 178) AS st_dma_id 
    #     , st.station_loc as feed
    #     , sch.vendor_name
    #     FROM detection.epg_schedule sch
    #     JOIN detection.local_national_simulcast_stations st -- add this table to detection schema
    #       ON st.station_id = sch.fk_station_id
    #       AND sch.vendor_name = st.vendor_name
    #     WHERE sch.airdate <= DATE_ADD(current_date(), 1) -- INTERVAL + 1 DAY
    #   )
    #   SELECT DISTINCT
    #   a.fk_show_id AS show_id
    #   , a.airdate AS airdate
    #   FROM main_feed a
    #   JOIN main_feed b
    #     ON a.fk_show_id = b.fk_show_id
    #     AND a.vendor_name = b.vendor_name
    #     AND a.airdate = b.airdate
    #     AND a.station_name != b.station_name
    #     AND CASE WHEN a.feed = 'Main' OR b.feed = 'Main' THEN TRUE
    #             WHEN a.natloc = 'national' OR b.natloc = 'national' THEN b.feed = a.feed
    #             WHEN a.natloc = 'local' AND b.natloc = 'local' THEN a.st_dma_id = b.st_dma_id OR a.st_dma_id = 178 OR b.st_dma_id = 178
    #        END
    # """)


    # simulcast_shows = deltaHelpers.saveToDeltaTempTable(simulcast_shows, "simulcast_shows")


    base_table = spark.sql(f"""--this snippet catures the data form content firehose for the previous hour
          SELECT DISTINCT 
                      vc.fk_station_id
                      , vc.fk_show_id
                      , vc.airdate
                      , vc.fk_tvid
                      , DATE_TRUNC('HOUR', vc.session_start) AS session_hour
          FROM detection.viewing_content_firehose vc --TO DO
          -- JOIN {catalog_temp}.{db_name_temp}.inscape_station_map_df ism
          --   ON ism.mapped_vendor_station_id = vc.fk_station_id
          --  AND ism.mapped_vendor = 'TIVO'
          -- JOIN {catalog_temp}.{db_name_temp}.simulcast_shows s
          --   ON vc.fk_show_id = s.show_id
          --  AND vc.airdate = s.airdate
          WHERE vc.session_start >= '{minBatchDate}'::timestamp  - interval '2 hour'
          AND vc.session_start < '{maxBatchDate}'::timestamp
          AND vc.fk_station_id IS NOT NULL  -- new addition
          """)

    base_table = deltaHelpers.saveToDeltaTempTable(base_table, "base_table")

    spark.sql(f"""
            -- Newest change 2022-01-21
            INSERT INTO {catalog_temp}.{db_name_temp}.base_table (
              SELECT DISTINCT vc.fk_station_id
                          , vc.fk_show_id
                          , vc.airdate
                          , vc.fk_tvid
                          , DATE_TRUNC('HOUR', vc.session_start) AS session_hour
              FROM {catalog_temp}.{db_name_temp}.step_1_df vc --content_firehose temp table in Redshift
            --   JOIN {catalog_temp}.{db_name_temp}.simulcast_shows s
            --     ON vc.fk_show_id = s.show_id
            --    AND vc.airdate = s.airdate
              WHERE vc.fk_station_id IS NOT NULL  -- new addition
            );
            """)

    # base_table = spark.sql(f"""SELECT * FROM {catalog_temp}.{db_name_temp}.base_table""") -- no need of this df

    acr_detected_df = spark.sql(f""" 
      -- Processing ACR detected sessions:
      -- Simulcast smoothing (station smooth when there is no show change)
      -- Input source smooth (input smooth when there is no show change)
      -- Apply live rule

      SELECT DISTINCT
      fk_tvid,
      fk_show_id,
      fk_station_id,
      airdate,
      session_start,
      session_end,
      datediff(second,session_start, session_end) session_duration,
      media_time_start,
      media_time_end,
      runtime ,
      fk_frame_id ,
      fk_dma_id,
      fk_zoo_id ,
      fk_content_id ,
      fk_input_source_id,
      fk_location_id ,
      fk_schedule_id ,
      timezone,
      confidence,
      ump_id,
      CASE
        WHEN fk_content_id <> 3468026
          AND (airdate - INTERVAL '120 Second') <= session_start
          AND session_start <= DATEADD(SECOND, 120 + runtime, airdate)
          AND session_end <= DATEADD(SECOND, 120 + runtime, airdate)
          AND session_duration <= (runtime + 120)
          AND DATEADD(SECOND, 120-media_time_start, session_start) >= airdate
          AND ABS(media_time_end - media_time_start - session_duration) <= 120
        THEN TRUE
        ELSE is_live
        END AS is_live,
      batch_size,
      file_ingested,
      created_at,
      audio_contri,
      video_contri,
      CASE WHEN timezone IS NOT NULL THEN convert_timezone('UTC', timezone,session_start) END AS local_session_start,
      CASE WHEN timezone IS NOT NULL THEN convert_timezone('UTC', timezone,session_end) END AS local_session_end,
      partition_key,
      input_file_name
      FROM (
          SELECT DISTINCT 
              c.fk_tvid as fk_tvid,
              c.fk_show_id,
              COALESCE(bt.fk_station_id, c.fk_station_id) AS fk_station_id,
              c.airdate,
              session_start,
              session_end,
              media_time_start,
              media_time_end,
              runtime,
              fk_frame_id,
              fk_dma_id,
              fk_zoo_id,
              fk_content_id,
              CASE WHEN c.fk_show_id = LAG(c.fk_show_id) OVER (PARTITION BY c.fk_tvid ORDER BY c.fk_tvid, c.session_start ASC)
                    AND c.fk_input_source_id != LAG(c.fk_input_source_id) OVER (PARTITION BY c.fk_tvid ORDER BY c.fk_tvid, c.session_start ASC)
                    THEN LAG(c.fk_input_source_id) OVER (PARTITION BY c.fk_tvid ORDER BY c.fk_tvid, c.session_start ASC)
              ELSE c.fk_input_source_id END AS fk_input_source_id,
              fk_location_id,
              fk_schedule_id,
              dm.timezone,
              c.is_live,
              c.file_ingested,
              c.confidence,
              c.ump_id,
              c.batch_size,
              c.audio_contri,
              c.video_contri,
              c.created_at,
              c.partition_key,
              c.input_file_name
          FROM {catalog_temp}.{db_name_temp}.step_1_df c
                    LEFT JOIN detection.location on location.location_id = c.fk_location_id
                    LEFT JOIN detection.dma dm ON dm.dma_id = location.fk_dma_id
                    LEFT JOIN {catalog_temp}.{db_name_temp}.base_table AS bt
                              ON c.fk_show_id = bt.fk_show_id
                                  AND c.fk_tvid = bt.fk_tvid
                                  AND c.airdate = bt.airdate
                                  AND (c.session_hour - INTERVAL 1 HOUR) = bt.session_hour
          )
      WHERE fk_content_id <> 3468026 OR fk_content_id IS NULL;
""")


    acr_detected_df = deltaHelpers.saveToDeltaTempTable(acr_detected_df, "acr_detected")


    acr_correction_df = spark.sql(f""" 
      -- ACR sessions that need to be corrected by Tuner data
      WITH overlapping AS (
          SELECT acr.fk_tvid,
                acr.session_start,
                acr.session_end,
                acr.session_duration,
                tun.fk_channel_id_tv2                                         AS tuner_channel_id,
                tun.fk_schedule_id_tv2                                        AS tuner_schedule_id,
                tun.fk_program_id_tv2                                         AS tuner_program_id,
                tun.tuner_channel_number                                      AS tuner_channel_number,
                GREATEST(tun.session_start, acr.session_start)                AS merged_session_start,
                LEAST(tun.session_end, acr.session_end)                       AS merged_session_end,
                DATEDIFF(second, merged_session_start, merged_session_end)    AS merged_duration
          FROM {catalog_temp}.{db_name_temp}.acr_detected acr
                  LEFT JOIN detection.epg_show
                            ON acr.fk_show_id = epg_show.show_id
                  JOIN detection.tuner_sessionized tun
                        ON acr.fk_tvid = tun.tvid
                            AND acr.fk_input_source_id IN (56, 96, 80) -- Warm (3468026)
                            --AND acr.fk_input_source_id IN (3, 16, 34) -- HOTC (42)
                            AND (GREATEST(tun.session_start, acr.session_start)
                                < LEAST(tun.session_end, acr.session_end))
                            AND lower(epg_show.title) = lower(tun.title_tv2)
                            AND acr.fk_station_id != tun.fk_channel_id_tv2
                            AND tun.data_type = 'tuner'
      )
      SELECT distinct fk_tvid,
            session_start,
            session_end,
            session_duration,
            tuner_channel_id,
            tuner_schedule_id,
            tuner_program_id,
            tuner_channel_number,
            SUM(merged_duration) AS diff_duration
      FROM overlapping
      GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
      HAVING diff_duration::float/session_duration::float > 0.5
      ;
        """)

    acr_correction_df = deltaHelpers.saveToDeltaTempTable(acr_correction_df, "acr_correction")

    new_tivo_tms_step = spark.sql(f"""
    -- Adding Tuner correction into ACR detected sessions
    SELECT 
      acr.fk_tvid,
      acr.fk_show_id,
      acr.fk_station_id,
      acr.session_duration,
      acr.airdate,
      acr.session_start,
      acr.session_end,
      acr.media_time_start,
      acr.media_time_end,
      acr.runtime,
      acr.fk_frame_id,
      acr.fk_dma_id,
      acr.fk_zoo_id,
      acr.fk_content_id,
      acr.fk_input_source_id,
      acr.fk_location_id,
      acr.fk_schedule_id,
      acr.timezone,
      acr.is_live,
      acr.file_ingested,
      acr.confidence,
      acr.ump_id,
      acr.batch_size,
      acr.audio_contri,
      acr.video_contri,
      acr.local_session_start,
      acr.local_session_end,
      COALESCE(corr.tuner_channel_id, NULL) AS tuner_channel_id,
      COALESCE(corr.tuner_schedule_id, NULL) AS tuner_schedule_id,
      COALESCE(corr.tuner_program_id, NULL) AS tuner_program_id,
      COALESCE(corr.tuner_channel_number, NULL) AS tuner_channel_number,
      acr.created_at,
      acr.partition_key,
      acr.input_file_name
    FROM {catalog_temp}.{db_name_temp}.acr_detected acr
    LEFT JOIN {catalog_temp}.{db_name_temp}.acr_correction corr
        ON acr.fk_tvid = corr.fk_tvid 
        AND acr.session_start = corr.session_start 
        AND acr.session_end = corr.session_end;
    """)
    
    deltaHelpers.saveToDeltaTempTable(new_tivo_tms_step, "new_tivo_tms_step")

    ### Lag Step
    df_lag_step = spark.sql(f"""
                SELECT 
                c.*,
                LAG(c.fk_show_id) OVER (
                              PARTITION BY c.fk_tvid
                              ORDER BY
                                c.fk_tvid,
                                c.session_start ASC
                            )
                            AS lag_show_id,

                LAG(c.fk_input_source_id) OVER (
                              PARTITION BY c.fk_tvid
                              ORDER BY
                                c.fk_tvid,
                                c.session_start ASC
                            )
                            AS lag_fk_input_source_id
                FROM {catalog_temp}.{db_name_temp}.step_1_df AS c
                """)

    df_lag_step = deltaHelpers.saveToDeltaTempTable(df_lag_step, "df_lag_step_cnt")

    ### tvevets_sessions Step

    tvevets_sessions_step = spark.sql(f"""
            select session_number,
                   fk_tvid,
                   case when session_type_new = 'gap' then null else channelid end as channelid,
                   case when session_type_new = 'gap' then null else programid end as programid,
                   case when session_type_new = 'gap' then null else airingid end as airingid,
                   airing_start,
                   case when coalesce(category_old,'ANTENNA') = 'ANTENNA' then null else category_old end category,
                   session_type_new as session_type,
                   max(environment) as environment,
                   min(eventdata_start_ts) eventdata_start_ts, max(eventdata_end_ts) eventdata_end_ts,
                   max(zoo) as zoo
            from (
                select *,
                       row_number() over (partition by fk_tvid, same_type order by eventdata_start_ts asc, eventdata_end_ts asc, programid asc) as rnum_same_type,
                       case when same_type = false then  rnum_same_type else  rnum - rnum_same_type  end typecase,
                       case when session_type_new = 'gap' then typecase when session_type_new = 'wf+' then -rnum end as session_number
                       from (
                            select fk_tvid, channelid, programid, airingid, airing_start,
                                   eventdata_start_ts, eventdata_end_ts,
                                   zoo,
                                   category as category_old,
                                   environment,
                                   case when category = 'ANTENNA' then 'gap' else session_type end session_type_new,
                                   coalesce(lag(case when category = 'ANTENNA' then 'gap' else session_type end) over (partition by fk_tvid order by eventdata_start_ts asc, eventdata_end_ts asc, programid asc)  = case when category = 'ANTENNA' then 'gap' else session_type end ,true) as same_type,
                                   row_number() over (partition by fk_tvid order by eventdata_start_ts asc, eventdata_end_ts asc, programid asc) as rnum,
                                   row_number() over (partition by fk_tvid, case when category = 'ANTENNA' then 'gap' else session_type end  order by eventdata_start_ts asc, eventdata_end_ts asc, programid asc) rnum_sess_type
                            from detection.tvevent_sessions
                            where eventdata_start_ts >= current_date   - interval '10 hours' 
                            )  wf
                ) wf
            group by 1,2,3,4,5,6,7,8                                      
                """)

    tvevets_sessions_step = deltaHelpers.saveToDeltaTempTable(tvevets_sessions_step, "tvevets_sessions_step")

    step_2 = spark.sql(f"""
    SELECT 
      c.fk_tvid as fk_tvid,
      c.fk_show_id AS fk_show_id,
      c.fk_station_id AS fk_station_id,
      c.airdate, 
      greatest(COALESCE(create_timestamp, c.session_start), c.session_start, eventdata_start_ts,tun.session_start) AS session_start, 
      least(eventdata_end_ts, next_create_timestamp, c.session_end,tun.session_end) AS session_end,  
      media_time_start,
      media_time_end,
      airing_start,
      tun.airdate_tv2 AS tuner_airdate,      
      runtime,
      fk_frame_id,
      fk_dma_id,
      fk_zoo_id,
      fk_content_id,
      COALESCE(i.fk_input_source_id, c.fk_input_source_id) AS fk_input_source_id,
      fk_location_id,
      fk_schedule_id,
      dm.timezone,
--      c.is_live,
      CASE WHEN c.fk_content_id = 3468026 AND (tun.fk_channel_id_tv2 IS NOT NULL OR tun.fk_schedule_id_tv2 IS NOT NULL OR tun.fk_program_id_tv2 IS NOT NULL) THEN true ELSE c.is_live END is_live, 
      c.file_ingested,
      c.confidence,
      c.ump_id,
      c.batch_size,
      c.audio_contri,
      c.video_contri,
      CASE WHEN COALESCE(ts.category,'') != 'ANTENNA' then ts.airingid else NULL END as vizio_epg_airing ,
      CASE WHEN COALESCE(ts.category,'') != 'ANTENNA' then ts.channelid else NULL END as vizio_epg_station ,
      CASE WHEN COALESCE(ts.category,'') != 'ANTENNA' then ts.programid else NULL END as vizio_epg_program ,
      tun.fk_channel_id_tv2  AS tuner_channel_id,    
      tun.fk_schedule_id_tv2 AS tuner_schedule_id,   
      tun.fk_program_id_tv2  AS tuner_program_id, 
      NVL2(tun.fk_channel_id_tv2,tun.tuner_channel_number,NULL) AS tuner_channel_number,
      c.created_at,
      c.partition_key,
      c.input_file_name
      FROM {catalog_temp}.{db_name_temp}.df_lag_step_cnt AS c
      LEFT JOIN (SELECT DISTINCT * FROM detection.location) l on l.location_id = c.fk_location_id --There are duplicates here
      LEFT JOIN (SELECT DISTINCT * FROM detection.dma) dm ON dm.dma_id = l.fk_dma_id -- There are dups here
      LEFT JOIN detection.tv_inputsource i ON --optimizer this by using file pruning if need be
          i.fk_tvid = c.fk_tvid
          AND i.create_timestamp < c.session_end
          AND i.next_create_timestamp > c.session_start
          AND i.next_create_timestamp > DATE_SUB(CURRENT_DATE(), 5)
      LEFT JOIN {catalog_temp}.{db_name_temp}.tvevets_sessions_step ts ON ts.fk_tvid = c.fk_tvid
          AND c.fk_content_id = 3468026
          AND c.session_start < ts.eventdata_end_ts
          AND c.session_end > ts.eventdata_start_ts
          AND i.create_timestamp < ts.eventdata_end_ts
          AND i.next_create_timestamp > ts.eventdata_start_ts
      AND i.app_name in ('OBFUSCATED','WatchFree+')
      AND i.fk_input_source_id IN (47, 32, 34425, 44, 48)
      AND (LOWER(ts.environment) = 'prod' OR ts.environment IS NULL)
      LEFT JOIN detection.tuner_sessionized tun 
        ON c.fk_tvid = tun.tvid
        AND c.fk_content_id = 3468026
        AND (COALESCE(i.fk_input_source_id, c.fk_input_source_id) IN  (56, 96, 80)
                        OR (COALESCE(i.fk_input_source_id, c.fk_input_source_id) IN (47, 32, 34425, 44, 48) and (ts.fk_tvid is null or ts.session_type = 'gap')))
        AND (GREATEST(ts.eventdata_start_ts, tun.session_start, COALESCE(create_timestamp, c.session_start),c.session_start)
                    < LEAST(ts.eventdata_end_ts, tun.session_end, COALESCE(next_create_timestamp, c.session_end), c.session_end))
     WHERE c.fk_content_id = 3468026
     """)

    step_2 = deltaHelpers.saveToDeltaTempTable(step_2, "step_2")

    # this used to be the final step. Adding one more step to calculate values for TMS and TiVo Tuner
    final_tivo_tms_step = spark.sql(f"""
        SELECT
        --DISTINCT --Not a great idea to run a distinct within a pipeline, usually this means more logic can be added, as this will only be a distinct per microBatch 
        fk_tvid,
        fk_show_id,
        fk_station_id,
        datediff(second,session_start, session_end) session_duration,
        COALESCE(airdate, airing_start, tuner_airdate) as airdate ,
        session_start,
        session_end,
        COALESCE(media_time_start, datediff(second, airing_start, session_start),datediff(second, tuner_airdate, session_start)) as media_time_start, 
        COALESCE(media_time_end, datediff(second, airing_start, session_end),datediff(second, tuner_airdate, session_end)) as media_time_end,
        runtime ,
        fk_frame_id ,
        fk_dma_id,
        fk_zoo_id ,
        fk_content_id ,
        fk_input_source_id,
        fk_location_id ,
        fk_schedule_id ,
        timezone,
        is_live,
        file_ingested,
        confidence, 
        ump_id,
        batch_size,
        audio_contri,
        video_contri,
        CASE WHEN timezone IS NOT NULL THEN FROM_UTC_TIMESTAMP(session_start, timezone) END AS local_session_start, 
        CASE WHEN timezone IS NOT NULL THEN FROM_UTC_TIMESTAMP(session_end, timezone) END AS local_session_end,
        vizio_epg_airing,
        vizio_epg_station,
        vizio_epg_program,
        tuner_channel_id,
        tuner_schedule_id,
        tuner_program_id,
        tuner_channel_number, 
        created_at,
        partition_key,
        input_file_name
      FROM {catalog_temp}.{db_name_temp}.step_2;
    """)

    deltaHelpers.saveToDeltaTempTable(final_tivo_tms_step, "final_tivo_tms_step")


    final_union = spark.sql(f"""
      SELECT 
        fk_tvid,
        fk_show_id,
        fk_station_id,
        session_duration,
        airdate,
        session_start,
        session_end,
        media_time_start,
        media_time_end,
        runtime,
        fk_frame_id,
        fk_dma_id,
        fk_zoo_id,
        fk_content_id,
        fk_input_source_id,
        fk_location_id,
        fk_schedule_id,
        timezone,
        is_live,
        file_ingested,
        confidence,
        ump_id,
        batch_size,
        audio_contri,
        video_contri,
        local_session_start,
        local_session_end,
        vizio_epg_airing,
        vizio_epg_station,
        vizio_epg_program,
        tuner_channel_id,
        tuner_schedule_id,
        tuner_program_id,
        tuner_channel_number,
        created_at,
        partition_key,
        input_file_name
      FROM
      {catalog_temp}.{db_name_temp}.final_tivo_tms_step

        UNION ALL

      SELECT 
        fk_tvid,
        fk_show_id,
        fk_station_id,
        session_duration,
        airdate,
        session_start,
        session_end,
        media_time_start,
        media_time_end,
        runtime,
        fk_frame_id,
        fk_dma_id,
        fk_zoo_id,
        fk_content_id,
        fk_input_source_id,
        fk_location_id,
        fk_schedule_id,
        timezone,
        is_live,
        file_ingested,
        confidence,
        ump_id,
        batch_size,
        audio_contri,
        video_contri,
        local_session_start,
        local_session_end,
        NULL AS vizio_epg_airing,
        NULL AS vizio_epg_station,
        NULL AS vizio_epg_program,
        tuner_channel_id,
        tuner_schedule_id,
        tuner_program_id,
        tuner_channel_number,
        created_at,
        partition_key,
        input_file_name
      FROM
      {catalog_temp}.{db_name_temp}.new_tivo_tms_step
    """)

    deltaHelpers.saveToDeltaTempTable(final_union, "final_union")
    

    #### Final Insert
    
    spark.sql(f"""
    INSERT INTO {catalog}.detection.viewing_content_firehose (
            fk_tvid,
            fk_show_id,
            fk_station_id,
            session_duration,
            airdate,
            session_start,
            session_end,
            media_time_start,
            media_time_end,
            runtime,
            fk_frame_id,
            fk_dma_id,
            fk_zoo_id,
            fk_content_id,
            fk_input_source_id,
            fk_location_id,
            fk_schedule_id,
            timezone,
            confidence,
            ump_id,
            is_live,
            batch_size,
            file_ingested,
            created_at,
            audio_contri,
            video_contri,
            local_session_start,
            local_session_end,
            vizio_epg_airing,
            vizio_epg_station,
            vizio_epg_program,
            tuner_channel_id,
            tuner_schedule_id,
            tuner_program_id,            
            tms_station_id,
            tms_show_id,
            tms_airdate,
            tms_schedule_id,
            tms_tuner_channel_id,
            tms_tuner_schedule_id,
            tms_tuner_program_id,
            tuner_channel_number,
            partition_key,
            input_file_name
          )
      SELECT
        vc.fk_tvid,
        vc.fk_show_id,
        tivo_ism.mapped_vendor_station_id AS fk_station_id,
        vc.session_duration,
        vc.airdate,
        vc.session_start,
        vc.session_end,
        vc.media_time_start, 
        vc.media_time_end,
        vc.runtime,
        vc.fk_frame_id ,
        vc.fk_dma_id,
        vc.fk_zoo_id ,
        vc.fk_content_id ,
        vc.fk_input_source_id,
        vc.fk_location_id ,
        vc.fk_schedule_id,
        vc.timezone,
        vc.confidence, 
        vc.ump_id,
        vc.is_live,
        vc.batch_size,
        vc.file_ingested, 
        vc.created_at,
        vc.audio_contri,
        vc.video_contri,
        vc.local_session_start, 
        vc.local_session_end,
        vc.vizio_epg_airing,
        vc.vizio_epg_station,
        vc.vizio_epg_program,
        tivo_tuner_sch_lat.fk_station_id AS tuner_channel_id,
        tivo_tuner_sch_lat.schedule_id   AS tuner_schedule_id,
        tivo_tuner_sch_lat.fk_show_id    AS tuner_program_id,
        vc.fk_station_id                  AS tms_station_id,
        tms_sch_lat.fk_show_id           AS tms_show_id,
        tms_sch_lat.airdate              AS tms_airdate,
        tms_sch.schedule_id              AS tms_schedule_id,
        vc.tuner_channel_id               AS tms_tuner_channel_id,
        vc.tuner_schedule_id              AS tms_tuner_schedule_id,
        vc.tuner_program_id               AS tms_tuner_program_id,
        vc.tuner_channel_number               AS tuner_channel_number,
        vc.partition_key,
        vc.input_file_name
      FROM {catalog_temp}.{db_name_temp}.final_union vc
      LEFT JOIN {catalog_temp}.{db_name_temp}.inscape_station_map_df tivo_ism
        ON tivo_ism.inscape_station_id = vc.fk_station_id
        AND tivo_ism.mapped_vendor = 'TIVO'
      LEFT JOIN {catalog_temp}.{db_name_temp}.epg_schedule_latest_df AS tms_sch_lat
        ON tms_sch_lat.fk_station_id = vc.fk_station_id
        AND tms_sch_lat.vendor_name = 'TMS'
        AND timestampadd(SECOND, vc.media_time_start, vc.airdate) > tms_sch_lat.airdate
        AND timestampadd(SECOND, vc.media_time_start, vc.airdate) <= tms_sch_lat.airdate_end
      LEFT JOIN detection.epg_schedule tms_sch
        ON tms_sch.airdate = tms_sch_lat.airdate
        AND tms_sch.fk_show_id = tms_sch_lat.fk_show_id
        AND tms_sch.fk_station_id = tms_sch_lat.fk_station_id
        AND DATE(tms_sch.airdate) >= CURRENT_DATE - INTERVAL 120 DAY
        AND tms_sch.airdate <= CURRENT_DATE + INTERVAL 2 DAY
      LEFT JOIN {catalog_temp}.{db_name_temp}.inscape_station_map_df AS tivo_tuner_map
        ON tivo_tuner_map.inscape_station_id = vc.tuner_channel_id
        AND tivo_tuner_map.mapped_vendor = 'TIVO'
      LEFT JOIN {catalog_temp}.{db_name_temp}.epg_schedule_latest_df AS tivo_tuner_sch_lat
          ON tivo_tuner_sch_lat.fk_station_id = tivo_tuner_map.mapped_vendor_station_id
        AND tivo_tuner_sch_lat.vendor_name = 'TIVO'
        AND timestampadd(SECOND, vc.media_time_start, vc.airdate) > tivo_tuner_sch_lat.airdate
        AND timestampadd(SECOND, vc.media_time_start, vc.airdate) <= tivo_tuner_sch_lat.airdate_end
      ORDER BY session_start,fk_tvid,fk_station_id,fk_show_id,fk_dma_id,fk_schedule_id;
    """)

   
    # moving optimize command to deletion process for opted out
    
    spark.sql(f"OPTIMIZE {catalog}.detection.viewing_content_firehose ZORDER BY (session_start, fk_tvid)")

    
    return

In [0]:
%sql
SELECT is_live, COUNT(*)
FROM prod.detection.viewing_content_firehose
WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 30 HOURS
  AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  AND file_ingested
GROUP BY 1